# Lesson 12.3 - LLMOps Foundations & LLM Lifecycle (architecture & lifecycle template)

This notebook demonstrates a lightweight LLMOps workflow without requiring external model downloads:

- prompt versioning,
- retrieval simulation,
- response generation via mock provider,
- evaluation scorecard and release gating.


## Objectives

1. Treat prompts and retrieval settings as versioned artifacts.
2. Evaluate quality, groundedness, and cost proxies.
3. Simulate release decisions for LLM workflow variants.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List
import re

import pandas as pd

## Step 1: Define Toy Knowledge Base and Retrieval

In [ ]:
KB = [
    {"id": "doc_1", "text": "Our enterprise plan supports SSO, audit logs, and priority support.", "tags": ["pricing", "enterprise"]},
    {"id": "doc_2", "text": "Starter plan includes up to 5 users and community support.", "tags": ["pricing", "starter"]},
    {"id": "doc_3", "text": "Data is encrypted at rest and in transit with role-based access control.", "tags": ["security", "compliance"]},
]


def simple_retrieve(query: str, top_k: int = 2) -> List[Dict[str, str]]:
    q_words = set(re.findall(r"[a-zA-Z]+", query.lower()))
    scored = []
    for doc in KB:
        doc_words = set(re.findall(r"[a-zA-Z]+", doc["text"].lower()))
        score = len(q_words.intersection(doc_words))
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0]

## Step 2: Prompt Registry and Mock LLM

In [ ]:
PROMPT_REGISTRY = {
    "v1": "Answer the question using the provided context. Be concise.",
    "v2": "Answer only from context. If uncertain, say 'I don't have enough evidence'. Include cited doc IDs.",
}

@dataclass
class LLMRun:
    run_id: str
    prompt_version: str
    query: str
    retrieved_docs: List[str]
    response: str
    grounded: int
    answer_length: int
    estimated_tokens: int


def mock_llm_answer(prompt: str, query: str, retrieved_docs: List[Dict[str, str]]) -> str:
    if not retrieved_docs:
        return "I don't have enough evidence."
    docs_text = " ".join([d["text"] for d in retrieved_docs])
    doc_ids = ", ".join([d["id"] for d in retrieved_docs])
    if "Include cited doc IDs" in prompt:
        return f"Based on context: {docs_text} [sources: {doc_ids}]"
    return f"Based on context: {docs_text}"

## Step 3: Run Two Prompt Variants

In [ ]:
queries = [
    "What security features do you provide?",
    "What does the starter plan include?",
    "Do you support enterprise audit logs?",
]

runs: List[LLMRun] = []

for i, q in enumerate(queries, start=1):
    retrieved = simple_retrieve(q, top_k=2)
    for pv in ["v1", "v2"]:
        response = mock_llm_answer(PROMPT_REGISTRY[pv], q, retrieved)
        grounded = int(bool(retrieved) and "Based on context" in response)
        est_tokens = len(response.split()) + len(q.split())
        runs.append(
            LLMRun(
                run_id=f"run_{i}_{pv}",
                prompt_version=pv,
                query=q,
                retrieved_docs=[d["id"] for d in retrieved],
                response=response,
                grounded=grounded,
                answer_length=len(response),
                estimated_tokens=est_tokens,
            )
        )

runs_df = pd.DataFrame([asdict(r) for r in runs])
runs_df.head()

## Step 4: Evaluation Scorecard and Release Gate

In [ ]:
scorecard = (
    runs_df.groupby("prompt_version")
    .agg(
        grounded_rate=("grounded", "mean"),
        avg_tokens=("estimated_tokens", "mean"),
        avg_answer_length=("answer_length", "mean"),
    )
    .reset_index()
)

# Example: prefer high grounded rate and lower token cost.
scorecard["composite_score"] = scorecard["grounded_rate"] - 0.003 * scorecard["avg_tokens"]
scorecard = scorecard.sort_values("composite_score", ascending=False)
scorecard

In [ ]:
release_candidate = scorecard.iloc[0]["prompt_version"]
release_reason = {
    "selected_version": release_candidate,
    "reason": "Highest composite score with groundedness and token efficiency balance.",
}
release_reason

## Production Case Studies & Exceptions

### Case 1: Support copilot with stale retrieval index
- Prompt quality looked stable, but answers degraded due to outdated index.
- Added index freshness checks and ingestion SLOs.

### Case 2: Cost spike from verbose responses
- Prompt update increased answer verbosity and token spend.
- Added response-length constraints and prompt regression tests.

### Case 3 (Exception): Deterministic workflow required
- In high-regulation workflows, team used LLM for draft generation only; final decision stayed rule-based.

## Interview Questions & Answers

1. **Why version prompts in LLMOps?**  
Prompt changes can materially alter behavior, quality, and cost.

2. **What is groundedness?**  
Degree to which output is supported by provided trusted context.

3. **How do you evaluate prompt versions?**  
Compare quality, safety, latency, and cost on a stable evaluation set.

4. **What is a prompt regression test?**  
A repeatable suite ensuring prompt updates don't break critical behaviors.

5. **How does retrieval quality affect LLM outputs?**  
Poor retrieval often causes confident but unsupported answers.

6. **Why monitor token usage?**  
Token volume is a major runtime cost driver.

7. **When should LLM output be constrained?**  
When compliance or deterministic formatting requirements are strict.

8. **What is LLM release gating?**  
Promotion based on quality/safety/cost thresholds.

9. **How do you handle no-evidence cases?**  
Use abstention logic and explicit uncertainty responses.

10. **Why combine automated and human evals?**  
Automated checks scale; human review captures nuanced quality/risk.